# Full-Range Flop-Only vs Full-Street Validation (GPU)

Runs the validation harness at **full ranges** on a GPU — the run the CPU couldn't do.

1. `Runtime -> Change runtime type -> GPU -> Save`
2. `Runtime -> Run all`. Code is embedded — nothing to upload.

The big run (~2–3 h for 12 boards, full ranges) writes results **after each board**, so a
disconnect still leaves partial data. Keep the tab active. The last cell prints `summary.json`
— **paste that back** and it becomes the updated findings.


In [ ]:
# 1) GPU present?
!nvidia-smi -L || echo 'NO GPU — Runtime -> Change runtime type -> GPU'

In [ ]:
# 2) Unpack embedded code (nothing to upload)
import base64, zipfile, io, os
_ZIP_B64 = (
    "..."
)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_ZIP_B64))) as z: z.extractall('poker')
print('unpacked ->', sorted(os.listdir('poker')))

In [ ]:
# 3) CuPy / GPU check
try:
    import cupy as cp
except Exception:
    import subprocess,sys; subprocess.run([sys.executable,'-m','pip','install','-q','cupy-cuda12x']); import cupy as cp
nm=cp.cuda.runtime.getDeviceProperties(0)['name']; nm=nm.decode() if isinstance(nm,bytes) else nm
print('cupy',cp.__version__,'| device:',nm)

In [ ]:
# 4) Smoke: one board on GPU (confirms the pipeline before the long run)
!cd poker && PYTHONPATH=src python -m pokertrainer.validate_flop --solver gpu --dtype float32 --n 400 --iters 120 --max-boards 1 --out /content/smoke
import json; print(json.load(open('/content/smoke/summary.json'))['totals'])

## The full run

Full ranges (`--n 400` uses every combo, ~250/side), `--iters 300`, float32, all 12 boards.
**~2–3 hours.** Progress prints per board; results stream to `/content/valout`.


In [ ]:
# 5) FULL-RANGE VALIDATION (long). Lower --max-boards for a faster first pass.
!cd poker && PYTHONPATH=src python -m pokertrainer.validate_flop \
    --solver gpu --dtype float32 --n 400 --iters 300 --max-boards 12 --out /content/valout

In [ ]:
# 6) Print summary.json — copy-paste this whole output back
import json; print(json.dumps(json.load(open('/content/valout/summary.json')), indent=1))

_The CSV is at `/content/valout/flop_validation.csv` (download via the left sidebar if needed). The pasted `summary.json` has every aggregate needed to update the findings._